# Library Chemical Space Explorer

**Purpose:** Decompose a compound library into structural archetypes for informed library synthesis decisions.

**Pipeline (in order):**
1. Compute 2D physicochemical descriptors from SMILES
2. Normalize with RobustScaler
3. Cluster in full descriptor space:
   - **K-Medoids** (cosine distance) → structural archetypes, each medoid is a real molecule
   - **HDBSCAN** (cosine distance) → natural density-based clusters, noise explicitly modeled
4. UMAP → 2D visualization canvas only (not used for clustering)
5. QC scores: Silhouette (K-Medoids), DBCV (HDBSCAN), UMAP stability ARI, Tanimoto intra-cluster diversity
6. Synthesis priority output: ranked medoid structures with activity profiles

**Inputs:**
- `library.csv` — your compound library (requires `smiles` column; optional: `name`, activity columns)
- `literature.csv` — small reference set of known actives (same format; optional)

**Outputs:**
- 6 figures (UMAP canvas × 4, Tanimoto heatmap, medoid structure grid)
- `synthesis_candidates.csv` — ranked medoid SMILES with cluster profiles
- `cluster_assignments.csv` — per-compound cluster labels + UMAP coordinates

In [ ]:
# ── Install dependencies (Colab) ──────────────────────────────────────────────
!pip install -q umap-learn hdbscan scikit-learn-extra rdkit-pypi
print('Dependencies installed.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit this cell before running
# ═══════════════════════════════════════════════════════════════════════════════

# --- File paths ---
LIBRARY_CSV    = 'library.csv'       # your compound library
LITERATURE_CSV = 'literature.csv'    # reference compounds (set to None to skip)

# --- Column names in your CSV ---
SMILES_COL     = 'smiles'            # SMILES column name
NAME_COL       = 'name'              # compound name/ID column (None if absent)
ACTIVITY_COL   = 'ic50_nm'          # IC50 or DC50 column (None to skip activity plot)
ACTIVITY_LABEL = 'IC50 (nM)'        # axis label for activity
ACTIVITY_LOG   = True               # log-transform activity? (True for IC50/DC50)

# --- Descriptors to use (6 recommended) ---
# Available: MolWt, MolLogP, TPSA, NumHAcceptors, NumHDonors,
#            NumRotatableBonds, RingCount, FractionCSP3, NumAromaticRings
DESCRIPTORS = [
    'MolWt',
    'MolLogP',
    'TPSA',
    'NumHAcceptors',
    'NumHDonors',
    'NumRotatableBonds',
]

# --- Clustering parameters ---
N_KMEDOIDS        = 8     # number of K-Medoids archetypes
HDBSCAN_MIN_SIZE  = 5     # min cluster size (lower = more clusters; tune to library size)
HDBSCAN_MIN_SAMP  = 3     # min samples (controls noise sensitivity)

# --- UMAP parameters (visualization only) ---
UMAP_N_NEIGHBORS  = 15    # lower = more local structure
UMAP_MIN_DIST     = 0.1
RANDOM_STATE      = 42

# --- Output ---
OUTPUT_DIR = 'results'

# ═══════════════════════════════════════════════════════════════════════════════
print('Configuration loaded.')

In [ ]:
import warnings
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

import umap
import hdbscan as hdbscan_lib
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn_extra.cluster import KMedoids

from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, AllChem, DataStructs
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import display, Image

warnings.filterwarnings('ignore')
Path(OUTPUT_DIR).mkdir(exist_ok=True)
print('Imports OK.')

## 1. Load Data & Compute Descriptors

In [ ]:
# ── Descriptor computation ────────────────────────────────────────────────────
_DESC_FN = {
    'MolWt':             Descriptors.MolWt,
    'MolLogP':           Descriptors.MolLogP,
    'TPSA':              Descriptors.TPSA,
    'NumHAcceptors':     Descriptors.NumHAcceptors,
    'NumHDonors':        Descriptors.NumHDonors,
    'NumRotatableBonds': Descriptors.NumRotatableBonds,
    'RingCount':         Descriptors.RingCount,
    'FractionCSP3':      Descriptors.FractionCSP3,
    'NumAromaticRings':  Descriptors.NumAromaticRings,
}

def compute_descriptors(smiles_list, descriptor_names):
    rows = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None:
            rows.append({d: np.nan for d in descriptor_names})
        else:
            rows.append({d: _DESC_FN[d](mol) for d in descriptor_names})
    return pd.DataFrame(rows)

def load_dataset(path, label, smiles_col, name_col, activity_col):
    df = pd.read_csv(path)
    df = df.rename(columns={smiles_col: 'smiles'})
    if name_col and name_col in df.columns:
        df = df.rename(columns={name_col: 'name'})
    else:
        df['name'] = [f'{label}_{i}' for i in range(len(df))]
    if activity_col and activity_col in df.columns:
        df = df.rename(columns={activity_col: 'activity'})
    df['source'] = label
    desc = compute_descriptors(df['smiles'].tolist(), DESCRIPTORS)
    df = pd.concat([df.reset_index(drop=True), desc], axis=1)
    before = len(df)
    df = df.dropna(subset=DESCRIPTORS)
    print(f'  {label}: {len(df):,} compounds ({before - len(df)} dropped — invalid SMILES)')
    return df

print('Descriptor functions ready.')

In [ ]:
# ── Load library ──────────────────────────────────────────────────────────────
lib_df = load_dataset(LIBRARY_CSV, 'library', SMILES_COL, NAME_COL, ACTIVITY_COL)

# ── Load literature reference (optional) ─────────────────────────────────────
if LITERATURE_CSV:
    lit_df = load_dataset(LITERATURE_CSV, 'literature', SMILES_COL, NAME_COL, ACTIVITY_COL)
    df = pd.concat([lib_df, lit_df], ignore_index=True)
else:
    df = lib_df.copy()
    print('  No literature reference provided — running library only.')

df = df.reset_index(drop=True)
print(f'\nTotal compounds: {len(df):,}')
print(f'Sources: {df["source"].value_counts().to_dict()}')
if 'activity' in df.columns:
    if ACTIVITY_LOG:
        df['activity_plot'] = np.log10(df['activity'].clip(lower=1e-3))
        print(f'Activity (log10): {df["activity_plot"].describe().round(2).to_dict()}')
    else:
        df['activity_plot'] = df['activity']
df.head(3)

## 2. Normalize

**RobustScaler** (median=0, IQR=1): resistant to outliers in MW and logP that would inflate StandardScaler's mean. No PCA — with ≤6 descriptors, PCA compresses physically interpretable axes into abstract components, making it impossible to ask *which descriptor drove this cluster*.

In [ ]:
X = df[DESCRIPTORS].values
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

print('Descriptor matrix shape:', X_scaled.shape)
print('Descriptors used:', DESCRIPTORS)
pd.DataFrame(X_scaled, columns=DESCRIPTORS).describe().round(3)

## 3. Cluster in Full Descriptor Space

Both algorithms operate on the normalized descriptor matrix — **not** on UMAP coordinates. UMAP runs later for visualization only.

In [ ]:
# ── Track A: K-Medoids (cosine distance) ─────────────────────────────────────
# Each medoid is a real compound — the most representative molecule in its
# region of descriptor space. These are your synthesis archetypes.
print('Running K-Medoids...')
km = KMedoids(
    n_clusters   = N_KMEDOIDS,
    metric       = 'cosine',
    method       = 'alternate',
    random_state = RANDOM_STATE,
)
km_labels  = km.fit_predict(X_scaled)
medoid_idx = list(km.medoid_indices_)

sil_km = silhouette_score(X_scaled, km_labels, metric='cosine')
print(f'K-Medoids done. Silhouette score (cosine): {sil_km:.4f}')
print(f'  > 0.5 = strong  |  0.25–0.5 = reasonable  |  < 0.25 = weak separation')

df['km_cluster'] = km_labels

# Print medoid summary
print('\nMedoid summary:')
for idx in medoid_idx:
    row = df.iloc[idx]
    act_str = f"  activity={row['activity']:.2f}" if 'activity' in df.columns and pd.notna(row.get('activity')) else ''
    print(f'  Cluster {km_labels[idx]}: {row["name"]}  source={row["source"]}{act_str}')

In [ ]:
# ── Track B: HDBSCAN (cosine distance) ───────────────────────────────────────
# Finds natural density-based clusters without forcing k.
# Noise points (label=-1) = compounds that don't belong to any dense region.
# DBCV (relative_validity_) is the correct QC metric for HDBSCAN.
# Silhouette assumes convex clusters; DBCV handles arbitrary shapes.
print('Running HDBSCAN...')
clusterer = hdbscan_lib.HDBSCAN(
    min_cluster_size       = HDBSCAN_MIN_SIZE,
    min_samples            = HDBSCAN_MIN_SAMP,
    metric                 = 'euclidean',      # on RobustScaler-normalized space
    cluster_selection_method = 'eom',
    gen_min_span_tree      = True,             # required for DBCV
)
hdb_labels = clusterer.fit_predict(X_scaled)

n_clusters = len(set(hdb_labels) - {-1})
n_noise    = (hdb_labels == -1).sum()
dbcv       = float(clusterer.relative_validity_)

print(f'HDBSCAN done.')
print(f'  Clusters: {n_clusters}  |  Noise: {n_noise} ({100*n_noise/len(df):.1f}%)')
print(f'  DBCV (relative_validity_): {dbcv:.4f}')
print(f'  > 0.5 = well-separated  |  0.0–0.5 = moderate  |  < 0 = poor')

df['hdb_cluster'] = hdb_labels

## 4. UMAP — Dimensionality Reduction for Visualization

UMAP compresses the normalized descriptor matrix to 2D XY coordinates for plotting. It is applied **after** clustering and its output is **never used as input to any algorithm**. Cluster assignments come from K-Medoids and HDBSCAN above.

UMAP stability check: run with 5 random seeds and compute pairwise Adjusted Rand Index (ARI) on cluster assignments. ARI ≥ 0.85 = layout is reproducible and cluster boundaries are stable.

In [ ]:
# ── UMAP stability check ──────────────────────────────────────────────────────
STABILITY_SEEDS   = [42, 1, 7, 99, 314]
STABILITY_ARI_MIN = 0.85

print(f'UMAP stability check ({len(STABILITY_SEEDS)} seeds)...')
seed_labels = []
seed_valid  = []
for seed in STABILITY_SEEDS:
    emb = umap.UMAP(
        n_neighbors  = UMAP_N_NEIGHBORS,
        min_dist     = UMAP_MIN_DIST,
        n_components = 2,
        metric       = 'cosine',
        random_state = seed,
    ).fit_transform(X_scaled)
    # Re-run HDBSCAN on the UMAP embedding to check layout stability
    lbl = hdbscan_lib.HDBSCAN(
        min_cluster_size=HDBSCAN_MIN_SIZE,
        min_samples=HDBSCAN_MIN_SAMP,
        gen_min_span_tree=True,
    ).fit_predict(emb)
    seed_labels.append(lbl)
    seed_valid.append(lbl != -1)

ari_vals = []
n = len(STABILITY_SEEDS)
for i in range(n):
    for j in range(i+1, n):
        shared = seed_valid[i] & seed_valid[j]
        if shared.sum() >= 10:
            ari_vals.append(adjusted_rand_score(seed_labels[i][shared], seed_labels[j][shared]))

if ari_vals:
    min_ari = float(np.min(ari_vals))
    print(f'  Min pairwise ARI: {min_ari:.3f}  (mean: {np.mean(ari_vals):.3f})')
    if min_ari >= STABILITY_ARI_MIN:
        print(f'  PASS — layout stable (>= {STABILITY_ARI_MIN})')
    else:
        print(f'  WARNING: layout unstable (< {STABILITY_ARI_MIN}) — increase UMAP_N_NEIGHBORS or HDBSCAN_MIN_SIZE')
else:
    min_ari = np.nan
    print('  Could not compute ARI — too few non-noise points across seeds.')

In [ ]:
# ── Main UMAP embedding (canonical seed=42) ───────────────────────────────────
print('Computing main UMAP embedding...')
reducer = umap.UMAP(
    n_neighbors  = UMAP_N_NEIGHBORS,
    min_dist     = UMAP_MIN_DIST,
    n_components = 2,
    metric       = 'cosine',
    random_state = RANDOM_STATE,
)
embedding = reducer.fit_transform(X_scaled)
df['umap_x'] = embedding[:, 0]
df['umap_y'] = embedding[:, 1]
print('UMAP done.')

## 5. Visualization

All plots share the same UMAP canvas. Cluster labels come from K-Medoids / HDBSCAN (descriptor space), not from UMAP coordinates.

In [ ]:
# ── Plot 1: Library vs Literature ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
colors = {'library': '#4C72B0', 'literature': '#DD8452'}
markers = {'library': 'o', 'literature': '*'}
sizes   = {'library': 8,  'literature': 120}

for src in df['source'].unique():
    mask = df['source'] == src
    ax.scatter(
        df.loc[mask, 'umap_x'], df.loc[mask, 'umap_y'],
        c=colors.get(src, 'grey'),
        marker=markers.get(src, 'o'),
        s=sizes.get(src, 8),
        alpha=0.6, label=f'{src} (n={mask.sum()})', zorder=3 if src=='literature' else 2,
        edgecolors='white' if src=='literature' else 'none', linewidths=0.5,
    )

ax.set_title('Chemical Space — Library vs Literature Reference', fontweight='bold')
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot1_source.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 1 saved.')

In [ ]:
# ── Plot 2: K-Medoids Archetypes ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
cmap = plt.cm.tab10

for lab in sorted(df['km_cluster'].unique()):
    mask = df['km_cluster'] == lab
    ax.scatter(
        df.loc[mask, 'umap_x'], df.loc[mask, 'umap_y'],
        c=[cmap(int(lab) % 10)], s=8, alpha=0.5,
        label=f'K{lab} (n={mask.sum()})', rasterized=True,
    )

# Mark medoids
ax.scatter(
    df.iloc[medoid_idx]['umap_x'], df.iloc[medoid_idx]['umap_y'],
    s=200, marker='*', c='black', zorder=10,
    edgecolors='white', linewidths=0.8, label='Medoids (synthesis candidates)',
)
for idx in medoid_idx:
    ax.annotate(
        f'K{km_labels[idx]}',
        (df.iloc[idx]['umap_x'], df.iloc[idx]['umap_y']),
        fontsize=7, ha='center', va='bottom',
        xytext=(0, 6), textcoords='offset points',
    )

ax.set_title(f'Track A — K-Medoids Archetypes (k={N_KMEDOIDS})\nSilhouette (cosine): {sil_km:.3f}', fontweight='bold')
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.legend(fontsize=7, ncol=2, loc='upper right')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot2_kmedoids.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 2 saved.')

In [ ]:
# ── Plot 3: HDBSCAN Natural Clusters ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
cmap_hdb = plt.cm.tab20

# Noise first (background)
noise_mask = df['hdb_cluster'] == -1
if noise_mask.sum():
    ax.scatter(
        df.loc[noise_mask, 'umap_x'], df.loc[noise_mask, 'umap_y'],
        c='lightgrey', s=5, alpha=0.3, label=f'Noise (n={noise_mask.sum()})', rasterized=True,
    )

for lab in sorted(set(df['hdb_cluster'].unique()) - {-1}):
    mask = df['hdb_cluster'] == lab
    ax.scatter(
        df.loc[mask, 'umap_x'], df.loc[mask, 'umap_y'],
        c=[cmap_hdb(int(lab) % 20)], s=10, alpha=0.7,
        label=f'C{lab} (n={mask.sum()})', rasterized=True,
    )

ax.set_title(f'Track B — HDBSCAN Natural Clusters ({n_clusters} clusters)\nDBCV: {dbcv:.3f}', fontweight='bold')
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.legend(fontsize=7, ncol=2, loc='upper right')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot3_hdbscan.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 3 saved.')

In [ ]:
# ── Plot 4: Activity Overlay ───────────────────────────────────────────────────
if 'activity_plot' in df.columns and df['activity_plot'].notna().sum() > 0:
    fig, ax = plt.subplots(figsize=(9, 6))

    has_act = df['activity_plot'].notna()

    # Grey out compounds without activity data
    ax.scatter(
        df.loc[~has_act, 'umap_x'], df.loc[~has_act, 'umap_y'],
        c='lightgrey', s=5, alpha=0.2, rasterized=True, label='No activity data',
    )

    vals = df.loc[has_act, 'activity_plot']
    # For IC50/DC50: lower = more potent → reverse colormap (green=low=potent)
    norm = mcolors.Normalize(vmin=np.percentile(vals, 5), vmax=np.percentile(vals, 95))
    sc = ax.scatter(
        df.loc[has_act, 'umap_x'], df.loc[has_act, 'umap_y'],
        c=vals, cmap='RdYlGn_r', norm=norm,
        s=10, alpha=0.7, rasterized=True,
    )
    cbar = plt.colorbar(sc, ax=ax)
    log_label = f'log10({ACTIVITY_LABEL})' if ACTIVITY_LOG else ACTIVITY_LABEL
    cbar.set_label(log_label)

    # Mark literature compounds
    if LITERATURE_CSV:
        lit_mask = (df['source'] == 'literature') & has_act
        ax.scatter(
            df.loc[lit_mask, 'umap_x'], df.loc[lit_mask, 'umap_y'],
            s=150, marker='*', c='black', zorder=10,
            edgecolors='white', linewidths=0.8, label=f'Literature (n={lit_mask.sum()})',
        )

    ax.set_title(f'Activity — {ACTIVITY_LABEL}\n(green = potent, red = inactive)', fontweight='bold')
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/plot4_activity.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Plot 4 saved.')
else:
    print('No activity data found — skipping Plot 4. Set ACTIVITY_COL in the config cell.')

## 6. Tanimoto Intra-Cluster Diversity

Mean pairwise Tanimoto similarity within each K-Medoids cluster (Morgan fingerprint, radius 2).

- **High similarity (> 0.7)** → cluster is a tight scaffold family; one representative is sufficient for synthesis
- **Low similarity (< 0.4)** → cluster spans diverse structures; sample multiple members

In [ ]:
from rdkit.Chem import AllChem

def morgan_fp(smi, radius=2, nbits=2048):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits)

print('Computing Morgan fingerprints...')
df['_fp'] = df['smiles'].apply(morgan_fp)
df_fp = df[df['_fp'].notna()].copy()

print('Computing intra-cluster Tanimoto similarity...')
cluster_tanimoto = []
for lab in sorted(df_fp['km_cluster'].unique()):
    fps = df_fp.loc[df_fp['km_cluster'] == lab, '_fp'].tolist()
    if len(fps) < 2:
        cluster_tanimoto.append({'cluster': lab, 'mean_tanimoto': np.nan, 'n': len(fps)})
        continue
    sims = []
    for i in range(min(len(fps), 200)):   # cap at 200 to avoid O(n²) on large clusters
        for j in range(i+1, min(len(fps), 200)):
            sims.append(DataStructs.TanimotoSimilarity(fps[i], fps[j]))
    cluster_tanimoto.append({
        'cluster': lab,
        'mean_tanimoto': round(float(np.mean(sims)), 3),
        'n': len(fps),
    })

tanimoto_df = pd.DataFrame(cluster_tanimoto)
print(tanimoto_df.to_string(index=False))

In [ ]:
# ── Plot 5: Tanimoto Diversity Heatmap ────────────────────────────────────────
# Pairwise mean Tanimoto between cluster medoids
medoid_fps = [morgan_fp(df.iloc[idx]['smiles']) for idx in medoid_idx]
medoid_fps_valid = [(i, fp) for i, fp in enumerate(medoid_fps) if fp is not None]

n_med = len(medoid_fps_valid)
sim_matrix = np.zeros((n_med, n_med))
for i, (_, fp_i) in enumerate(medoid_fps_valid):
    for j, (_, fp_j) in enumerate(medoid_fps_valid):
        sim_matrix[i, j] = DataStructs.TanimotoSimilarity(fp_i, fp_j)

labels_heatmap = [f'K{km_labels[medoid_idx[i]]}' for i, _ in medoid_fps_valid]

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Tanimoto Similarity')
ax.set_xticks(range(n_med)); ax.set_yticks(range(n_med))
ax.set_xticklabels(labels_heatmap, rotation=45, ha='right')
ax.set_yticklabels(labels_heatmap)
for i in range(n_med):
    for j in range(n_med):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=8)
ax.set_title('Medoid Pairwise Tanimoto Similarity\n(low = diverse clusters, high = redundant)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot5_tanimoto_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 5 saved.')

In [ ]:
# ── Plot 6: Medoid Structure Grid ─────────────────────────────────────────────
# 2D structures of all K-Medoid archetypes — these are your synthesis candidates
medoid_mols   = []
medoid_labels = []

for idx in medoid_idx:
    row = df.iloc[idx]
    mol = Chem.MolFromSmiles(str(row['smiles']))
    if mol:
        AllChem.Compute2DCoords(mol)
        medoid_mols.append(mol)
        act_str = ''
        if 'activity' in df.columns and pd.notna(row.get('activity')):
            act_str = f'\n{ACTIVITY_LABEL}={row["activity"]:.1f}'
        medoid_labels.append(
            f'K{km_labels[idx]} | {row["source"]}\n{row["name"]}{act_str}'
        )

img = Draw.MolsToGridImage(
    medoid_mols,
    molsPerRow    = 4,
    subImgSize    = (300, 250),
    legends       = medoid_labels,
    returnPNG     = False,
)
img.save(f'{OUTPUT_DIR}/plot6_medoid_structures.png')
display(img)
print('Plot 6 saved — these are your synthesis archetypes.')

## 7. Synthesis Priority Output

Clusters ranked by:
1. Literature overlap → validated chemical space (highest confidence)
2. Library-only clusters with low intra-cluster Tanimoto → novel diverse scaffolds
3. Noise or singleton regions → deprioritize

In [ ]:
rows = []
for idx in medoid_idx:
    lab  = int(km_labels[idx])
    mask = df['km_cluster'] == lab
    row  = df.iloc[idx]

    n_lib = (df.loc[mask, 'source'] == 'library').sum()
    n_lit = (df.loc[mask, 'source'] == 'literature').sum() if LITERATURE_CSV else 0

    if n_lit > 0 and n_lib > 0:
        coverage = 'overlap'
    elif n_lit > 0:
        coverage = 'literature_only'
    else:
        coverage = 'library_only'

    tan = tanimoto_df.loc[tanimoto_df['cluster'] == lab, 'mean_tanimoto'].values
    tan_val = float(tan[0]) if len(tan) else np.nan

    act_med = np.nan
    if 'activity' in df.columns:
        act_vals = df.loc[mask, 'activity'].dropna()
        if len(act_vals):
            act_med = float(np.median(act_vals))

    # Descriptor profile
    sub = df.loc[mask]
    desc_profile = {d: round(float(sub[d].median()), 1) for d in DESCRIPTORS}

    rows.append({
        'cluster':          lab,
        'medoid_name':      row['name'],
        'medoid_smiles':    row['smiles'],
        'medoid_source':    row['source'],
        'n_total':          int(mask.sum()),
        'n_library':        int(n_lib),
        'n_literature':     int(n_lit),
        'coverage':         coverage,
        'mean_tanimoto':    tan_val,
        'median_activity':  round(act_med, 2) if not np.isnan(act_med) else None,
        **{f'median_{k}': v for k, v in desc_profile.items()},
    })

synthesis_df = pd.DataFrame(rows)

# Rank: overlap first, then library_only sorted by diversity (low Tanimoto = diverse = prioritize)
coverage_order = {'overlap': 0, 'library_only': 1, 'literature_only': 2}
synthesis_df['_rank'] = synthesis_df['coverage'].map(coverage_order)
synthesis_df = synthesis_df.sort_values(['_rank', 'mean_tanimoto']).drop(columns='_rank')
synthesis_df.to_csv(f'{OUTPUT_DIR}/synthesis_candidates.csv', index=False)

print('Synthesis priority table:')
display(synthesis_df[['cluster','medoid_name','coverage','n_total','mean_tanimoto','median_activity']].to_string(index=False))
print(f'\nFull table saved: {OUTPUT_DIR}/synthesis_candidates.csv')

In [ ]:
# ── Save per-compound cluster assignments ─────────────────────────────────────
out_cols = ['name', 'smiles', 'source', 'km_cluster', 'hdb_cluster', 'umap_x', 'umap_y']
if 'activity' in df.columns:
    out_cols.append('activity')
out_cols += DESCRIPTORS

df[out_cols].to_csv(f'{OUTPUT_DIR}/cluster_assignments.csv', index=False)
print(f'Per-compound assignments saved: {OUTPUT_DIR}/cluster_assignments.csv')

## 8. QC Summary

In [ ]:
print('=' * 55)
print('QC SUMMARY')
print('=' * 55)
print(f'Compounds analyzed         : {len(df):,}')
print(f'Descriptors used           : {DESCRIPTORS}')
print()
print(f'K-Medoids (k={N_KMEDOIDS})')
print(f'  Silhouette score (cosine): {sil_km:.4f}')
print(f'  Interpretation           : {"strong" if sil_km > 0.5 else "reasonable" if sil_km > 0.25 else "weak"}')
print()
print(f'HDBSCAN')
print(f'  Clusters found           : {n_clusters}')
print(f'  Noise points             : {n_noise} ({100*n_noise/len(df):.1f}%)')
print(f'  DBCV                     : {dbcv:.4f}')
print(f'  Interpretation           : {"well-separated" if dbcv > 0.5 else "moderate" if dbcv > 0 else "poor"}')
print()
print(f'UMAP stability')
if not np.isnan(min_ari):
    print(f'  Min pairwise ARI         : {min_ari:.4f}')
    print(f'  Status                   : {"PASS" if min_ari >= STABILITY_ARI_MIN else "FAIL — increase n_neighbors"}')
else:
    print(f'  Status                   : could not compute')
print('=' * 55)